# 72 — Batch Agent Runner
## Process Many Tasks at Once — Async Concurrency for Agent Workflows
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/72-batch-agent-runner/batch_agent_runner_workbook.ipynb)

When your agent has ten tasks to do, does it finish them one-at-a-time — or all at once? Sequential LLM calls are the default path of least resistance, but they waste time proportional to the number of tasks. This workshop teaches **async batch processing**: grouping tasks into controlled-size batches, firing them concurrently with `asyncio.gather`, and retrying failures automatically with exponential backoff. The result is a durable, rate-limit-aware runner that completes N independent tasks in roughly the time a single task takes.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Why batching? Sequential vs concurrent wall-clock time |
| 2 | **Setup** — Install deps, load API key |
| 3 | **asyncio Primer** — `async def`, `await`, `asyncio.gather` |
| 4 | **`ainvoke` vs `invoke`** — Async LLM calls with LangChain |
| 5 | **Constants and Task List** — `BATCH_SIZE`, `MAX_RETRIES`, `RETRY_DELAY` |
| 6 | **`process_task()`** — One task: invoke, catch errors, retry with backoff |
| 7 | **Exponential Backoff** — Why it works and how to tune it |
| 8 | **Batching Logic** — Split tasks into chunks; gather within each batch |
| 9 | **`run_batch_agent()`** — Full orchestrator: batch loop + gather |
| 10 | **Timing Experiment** — Measure sequential vs async speedup |
| 11 | **Results Inspection** — Parse statuses, count errors, display answers |
| 12 | **Tuning BATCH_SIZE** — Trade-off: throughput vs rate limits |
| 13 | **Error Injection** — Simulate failures; watch retry + backoff in action |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- No external files needed — all tasks are defined inline

### Key References
> - Python asyncio docs: https://docs.python.org/3/library/asyncio.html
> - LangChain async guide: https://python.langchain.com/docs/concepts/async/
> - Exponential backoff (AWS best practice): https://docs.aws.amazon.com/general/latest/gr/api-retries.html

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain", "langchain-openai", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 1 — Why Batch Processing?

When you have N independent LLM tasks, you have two choices:

**Sequential execution:**
```
task1 → [wait 1s] → task2 → [wait 1s] → task3 → ... → taskN
Total time ≈ N × avg_latency
```

**Concurrent execution (batched):**
```
[task1, task2, task3] → asyncio.gather → all done in ~max(latencies)
→ [task4, task5, task6] → asyncio.gather → ...
Total time ≈ (N / BATCH_SIZE) × avg_latency
```

With 9 tasks and BATCH_SIZE=3, you get **3 batches** each containing 3 parallel calls. If each call takes 1 second, sequential takes ~9 seconds but batched takes ~3 seconds — a **3× speedup**.

### Why not run everything at once?

| Strategy | Throughput | Rate limit risk | Complexity |
|----------|-----------|-----------------|------------|
| Sequential | Slowest | None | Trivial |
| All concurrent | Fastest | High (burst) | Medium |
| **Batched (this pattern)** | **Fast** | **Controlled** | **Low** |

Firing all tasks simultaneously risks hitting OpenAI's **Requests Per Minute (RPM)** or **Tokens Per Minute (TPM)** limits, causing 429 errors. Batching caps inflight requests at `BATCH_SIZE`, giving you control over burst width.

## Part 2 — asyncio Primer

Python's `asyncio` library lets a single thread manage multiple I/O-bound operations concurrently. The key primitives:

```python
async def my_func():    # declares a coroutine — does not run yet
    result = await some_io()  # suspends here; event loop runs other coroutines
    return result

asyncio.run(my_func())  # creates event loop and runs the coroutine

# Run N coroutines concurrently — returns list of results in order
results = await asyncio.gather(coro1(), coro2(), coro3())
```

**`asyncio.gather` is the key primitive here.** It starts all coroutines simultaneously and awaits all of them, returning results in the same order as the inputs — even if they finish in a different order.

### Concurrency vs Parallelism

> **Concurrency**: one thread, multiple tasks interleaved — tasks take turns at await points  
> **Parallelism**: multiple threads/processes running truly simultaneously

LLM calls are *I/O-bound* (waiting for network), so concurrency is sufficient. No threads or multiprocessing needed.

In [ ]:
# Demonstrate asyncio.gather with a trivial example before touching LLMs
import asyncio
import time

async def fake_io(label: str, delay: float) -> str:
    """Simulates an I/O call that takes `delay` seconds."""
    await asyncio.sleep(delay)
    return f"{label} done after {delay}s"

async def demo_sequential():
    t0 = time.time()
    r1 = await fake_io("task1", 0.3)
    r2 = await fake_io("task2", 0.3)
    r3 = await fake_io("task3", 0.3)
    elapsed = time.time() - t0
    print(f"Sequential: {elapsed:.2f}s — {[r1, r2, r3]}")

async def demo_concurrent():
    t0 = time.time()
    results = await asyncio.gather(
        fake_io("task1", 0.3),
        fake_io("task2", 0.3),
        fake_io("task3", 0.3),
    )
    elapsed = time.time() - t0
    print(f"Concurrent: {elapsed:.2f}s — {results}")

await demo_sequential()
await demo_concurrent()

**You should see ~0.90s sequential vs ~0.30s concurrent** — a 3× speedup with zero threads. The concurrent version runs all three sleeps simultaneously because `await asyncio.sleep()` yields control back to the event loop.

> **Jupyter note:** In Jupyter, the event loop is already running, so you can `await` directly in cells without `asyncio.run()`. In scripts (like `main.py`), you need `asyncio.run()`.

## Part 3 — `ainvoke` vs `invoke`

LangChain exposes two interfaces for calling an LLM:

| Method | Type | Use when |
|--------|------|----------|
| `llm.invoke(messages)` | Synchronous | Simple scripts, sequential calls |
| `await llm.ainvoke(messages)` | Asynchronous | Inside `async def`, parallel batches |

Both accept the same arguments and return the same `AIMessage` response. The async version suspends at the network call, letting `asyncio.gather` run other coroutines while waiting.

```python
# Sync — blocks the thread until response arrives
response = llm.invoke([HumanMessage(content="Hello")])

# Async — suspends this coroutine; event loop can run others
response = await llm.ainvoke([HumanMessage(content="Hello")])
```

The batch runner uses `ainvoke` because each `process_task` coroutine needs to be runnable concurrently inside `asyncio.gather`.

In [ ]:
# Quick demonstration: ainvoke returns the same AIMessage as invoke
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Single async call — verify ainvoke works in this environment
response = await llm.ainvoke([HumanMessage(content="Say 'batch runner ready' and nothing else.")])
print(f"Response type: {type(response).__name__}")
print(f"Content: {response.content}")

## Part 4 — Constants and the Task List

The batch runner has three tunable constants that control throughput and resilience:

| Constant | Default | Effect |
|----------|---------|--------|
| `BATCH_SIZE` | 3 | Max concurrent requests per batch. Raise for more throughput; lower to avoid 429s |
| `MAX_RETRIES` | 3 | Attempts per task before giving up with `"error"` status |
| `RETRY_DELAY` | 1.0 | Base delay (seconds) for exponential backoff. Actual delay = `RETRY_DELAY × 2^attempt` |

The **task list** is just a list of strings — each string is a prompt sent to the LLM. Tasks must be **independent** (no task depends on another's output). If tasks have dependencies, use a sequential pipeline or a DAG instead.

In [ ]:
# Constants from src/tools.py
BATCH_SIZE = 3
MAX_RETRIES = 3
RETRY_DELAY = 1.0

SAMPLE_TASKS = [
    "What is photosynthesis? Answer in one sentence.",
    "What is the capital of Japan? Answer in one sentence.",
    "What is Newton's first law? Answer in one sentence.",
    "What is DNA? Answer in one sentence.",
    "What is the speed of light? Answer in one sentence.",
    "What is machine learning? Answer in one sentence.",
    "What is blockchain? Answer in one sentence.",
    "What is quantum computing? Answer in one sentence.",
    "What is the Pythagorean theorem? Answer in one sentence.",
]

# Show batch plan before running
batches = [SAMPLE_TASKS[i:i + BATCH_SIZE] for i in range(0, len(SAMPLE_TASKS), BATCH_SIZE)]
print(f"{len(SAMPLE_TASKS)} tasks → {len(batches)} batches of {BATCH_SIZE}")
for bi, batch in enumerate(batches):
    print(f"  Batch {bi+1}: {[t.split('?')[0] for t in batch]}")

## Part 5 — `process_task()`: One Task with Retry

`process_task` is the atomic unit. It handles a **single task** with full error recovery:

```
process_task(llm, task, index)
  ┌─ attempt 0 ─────────────────────────────────┐
  │  await llm.ainvoke(...)                      │
  │    → success: return {status: "ok", ...}     │
  │    → error:                                  │
  │       attempt < MAX_RETRIES-1?               │
  │         yes → sleep(RETRY_DELAY × 2^attempt) │
  │              → attempt 1 → attempt 2 → ...   │
  │         no  → return {status: "error", ...}  │
  └──────────────────────────────────────────────┘
```

The return value is a **dict** with a consistent schema:
```python
{"index": int, "task": str, "answer": str, "status": "ok" | "error: ...", "attempts": int}
```

Using a dict (rather than raising exceptions) lets the batch runner collect all results — successes and failures alike — without one failure cancelling the others.

In [ ]:
# process_task from src/tools.py
import asyncio
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

async def process_task(llm: ChatOpenAI, task: str, index: int) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            response = await llm.ainvoke([HumanMessage(content=task)])
            return {
                "index": index,
                "task": task,
                "answer": response.content.strip(),
                "status": "ok",
                "attempts": attempt + 1,
            }
        except Exception as exc:
            if attempt < MAX_RETRIES - 1:
                await asyncio.sleep(RETRY_DELAY * (2 ** attempt))
            else:
                return {
                    "index": index,
                    "task": task,
                    "answer": "",
                    "status": f"error: {exc}",
                    "attempts": attempt + 1,
                }
    return {"index": index, "task": task, "answer": "", "status": "error", "attempts": MAX_RETRIES}

# Run one task standalone to verify
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
result = await process_task(llm, "What is photosynthesis? Answer in one sentence.", index=0)
print(result)

## Part 6 — Exponential Backoff

When a request fails (network error, 429 rate limit, 500 server error), retrying immediately often hits the same error again. **Exponential backoff** adds increasing wait times between retries:

| Attempt | Wait before retry |
|---------|------------------|
| 0 (first try) | — |
| 1 | `RETRY_DELAY × 2^0` = 1.0s |
| 2 | `RETRY_DELAY × 2^1` = 2.0s |
| 3 (final) | return error (no more retries) |

The total wait before giving up is 1 + 2 = **3 seconds** with the defaults.

**Why exponential?** When the API is overloaded, all clients retrying at the same fixed interval create synchronized bursts (thundering herd). Exponential growth means different clients spread out their retries naturally.

In production, add **jitter** (random offset) to further desynchronize retries:
```python
import random
delay = RETRY_DELAY * (2 ** attempt) + random.uniform(0, 0.5)
```

In [ ]:
# Visualize backoff schedule for different configs
def backoff_schedule(retry_delay: float, max_retries: int) -> list:
    waits = []
    for attempt in range(max_retries - 1):  # no wait after final attempt
        waits.append(retry_delay * (2 ** attempt))
    return waits

configs = [
    ("default", 1.0, 3),
    ("aggressive", 0.5, 4),
    ("conservative", 2.0, 5),
]

for name, delay, retries in configs:
    schedule = backoff_schedule(delay, retries)
    total_wait = sum(schedule)
    print(f"{name:12s} (delay={delay}, retries={retries}): waits={schedule} total_wait={total_wait:.1f}s")

## Part 7 — Batching Logic

The core of the batch runner is splitting the task list into fixed-size chunks and processing each chunk with `asyncio.gather`.

**Splitting into batches:**
```python
tasks = [t0, t1, t2, t3, t4, t5, t6, t7, t8]  # 9 tasks
BATCH_SIZE = 3

batches = [tasks[i:i+BATCH_SIZE] for i in range(0, len(tasks), BATCH_SIZE)]
# → [[t0,t1,t2], [t3,t4,t5], [t6,t7,t8]]
```

**Running one batch:**
```python
batch_results = await asyncio.gather(
    process_task(llm, t0, 0),
    process_task(llm, t1, 1),
    process_task(llm, t2, 2),
)
# All three run concurrently; gather returns when the last one finishes
```

**Index tracking:** Notice `i + batch_idx * BATCH_SIZE` in the workflow. This ensures the `index` field in each result dict matches the task's position in the original list, not its position within the current batch.

In [ ]:
# Show the batch splitting logic in isolation
def split_into_batches(tasks: list, batch_size: int) -> list:
    return [tasks[i:i + batch_size] for i in range(0, len(tasks), batch_size)]

# Demonstrate index tracking across batches
tasks = [f"task_{i}" for i in range(9)]
batches = split_into_batches(tasks, BATCH_SIZE)

print("Batch structure and global indices:")
for batch_idx, batch in enumerate(batches):
    print(f"  Batch {batch_idx + 1}:")
    for i, task in enumerate(batch):
        global_idx = i + batch_idx * BATCH_SIZE
        print(f"    local_i={i} → global_idx={global_idx}: {task}")

## Part 8 — `run_batch_agent()`: Full Orchestrator

The orchestrator ties everything together:

```
run_batch_agent(tasks)
  │
  ├─ create LLM client (shared across all tasks)
  ├─ split tasks into batches
  └─ for each batch:
       ├─ print progress
       ├─ asyncio.gather(process_task × BATCH_SIZE)
       └─ extend results list
  └─ return results (list of dicts, same order as input tasks)
```

Key design decisions:
- **Single LLM instance**: `ChatOpenAI` is created once and shared. It is stateless and thread-safe for async use.
- **Sequential batches**: `for batch in batches` runs batches one after another. Only tasks *within* a batch run concurrently.
- **`results.extend(batch_results)`**: Accumulates results across batches into one flat list.
- **Order preservation**: `asyncio.gather` returns results in input order, so the final list maintains the original task ordering.

In [ ]:
# run_batch_agent from src/workflow.py
from langchain_openai import ChatOpenAI

async def run_batch_agent(tasks: list[str]) -> list[dict]:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    results: list[dict] = []
    batches = [tasks[i:i + BATCH_SIZE] for i in range(0, len(tasks), BATCH_SIZE)]

    for batch_idx, batch in enumerate(batches):
        print(f"  Running batch {batch_idx + 1}/{len(batches)} ({len(batch)} tasks)...")
        batch_results = await asyncio.gather(
            *[process_task(llm, task, i + batch_idx * BATCH_SIZE)
              for i, task in enumerate(batch)]
        )
        results.extend(batch_results)

    return results

print("run_batch_agent defined — ready to run.")

## Part 9 — Run the Full Batch Agent

Now we run all 9 tasks through the batch agent. Watch the batch progress lines — each batch fires 3 concurrent requests and completes before the next batch starts.

**Expected output pattern:**
```
Running 9 tasks in batches of 3
  Running batch 1/3 (3 tasks)...
  Running batch 2/3 (3 tasks)...
  Running batch 3/3 (3 tasks)...

Completed: 9/9 ok, 0 errors
  [01] OK  Photosynthesis is the process...
  ...
```

In [ ]:
import time

print(f"Running {len(SAMPLE_TASKS)} tasks in batches of {BATCH_SIZE}\n")

t0 = time.time()
results = await run_batch_agent(SAMPLE_TASKS)
elapsed = time.time() - t0

ok = [r for r in results if r["status"] == "ok"]
errors = [r for r in results if r["status"] != "ok"]
print(f"\nCompleted: {len(ok)}/{len(results)} ok, {len(errors)} errors")
print(f"Wall-clock time: {elapsed:.2f}s ({elapsed/len(results):.2f}s avg per task)\n")

for r in results:
    status = "OK" if r["status"] == "ok" else r["status"]
    print(f"  [{r['index'] + 1:02d}] {status}  {r['answer'][:80]}")

## Part 10 — Timing Experiment: Sequential vs Async

Let's measure the actual speedup of async batching vs sequential execution on the same 9 tasks.

In [ ]:
import time

llm_sync = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Sequential baseline ---
print("Sequential execution:")
t_seq_start = time.time()
seq_results = []
for i, task in enumerate(SAMPLE_TASKS):
    r = await llm_sync.ainvoke([HumanMessage(content=task)])
    seq_results.append(r.content.strip())
    print(f"  [{i+1:02d}] done")
t_seq = time.time() - t_seq_start

print(f"\nSequential total: {t_seq:.2f}s")

# --- Async batch ---
print("\nAsync batch execution:")
t_batch_start = time.time()
batch_results = await run_batch_agent(SAMPLE_TASKS)
t_batch = time.time() - t_batch_start

print(f"\nAsync batch total: {t_batch:.2f}s")
print(f"\nSpeedup: {t_seq / t_batch:.1f}x faster with BATCH_SIZE={BATCH_SIZE}")
print(f"(Theoretical max speedup ≈ {BATCH_SIZE}x)")

## Part 11 — Results Inspection

The results list is a structured dataset. Each element is a dict with:
- `index` — original position in the task list (0-based)
- `task` — the input prompt
- `answer` — LLM response content (empty string if error)
- `status` — `"ok"` or `"error: <message>"`
- `attempts` — how many attempts were needed (1 = first try succeeded)

This structured output makes it easy to:
- Separate successes from failures for downstream processing
- Retry only the failed tasks
- Log or persist results to a file or database

In [ ]:
# Inspect the results structure
print("=== First result dict ===")
import json
# Truncate answer for display
display_result = {**results[0], "answer": results[0]["answer"][:60] + "..."}
print(json.dumps(display_result, indent=2))

print("\n=== Summary stats ===")
statuses = [r["status"] for r in results]
attempts = [r["attempts"] for r in results]
print(f"Total tasks:    {len(results)}")
print(f"Succeeded:      {statuses.count('ok')}")
print(f"Failed:         {len([s for s in statuses if s != 'ok'])}")
print(f"First-try wins: {attempts.count(1)} (needed 0 retries)")
print(f"Max attempts:   {max(attempts)}")

print("\n=== Failed tasks (if any) ===")
failed = [r for r in results if r["status"] != "ok"]
if failed:
    for r in failed:
        print(f"  [{r['index']+1:02d}] {r['status']} after {r['attempts']} attempts")
else:
    print("  None — all tasks succeeded.")

## Part 12 — Tuning `BATCH_SIZE`

`BATCH_SIZE` is the primary knob for tuning throughput vs rate-limit safety.

| BATCH_SIZE | Concurrency | Rate limit risk | Latency |
|------------|------------|-----------------|--------|
| 1 | None (sequential) | Zero | N × avg |
| 3 | Low | Low | ≈ (N/3) × avg |
| 10 | Medium | Medium | ≈ (N/10) × avg |
| N (all at once) | High | High | ≈ max(all) |

**OpenAI rate limits (gpt-4o-mini, free tier):**
- 500 RPM (requests per minute)
- 200,000 TPM (tokens per minute)

For a typical task using ~100 tokens, you can safely run up to ~50 concurrent requests before hitting TPM. `BATCH_SIZE=3` is deliberately conservative for teaching purposes.

> **Rule of thumb:** Start at BATCH_SIZE=5. If you see 429 errors, halve it. If everything's fine, double it until you find the limit.

In [ ]:
# Compare wall-clock time for BATCH_SIZE = 1, 3, 9
async def run_with_batch_size(tasks: list[str], batch_size: int) -> tuple[float, list[dict]]:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    results: list[dict] = []
    batches = [tasks[i:i + batch_size] for i in range(0, len(tasks), batch_size)]
    t0 = time.time()
    for batch_idx, batch in enumerate(batches):
        batch_results = await asyncio.gather(
            *[process_task(llm, task, i + batch_idx * batch_size)
              for i, task in enumerate(batch)]
        )
        results.extend(batch_results)
    return time.time() - t0, results

print("Batch size comparison (using 6 tasks to keep cost low):")
six_tasks = SAMPLE_TASKS[:6]

for bs in [1, 3, 6]:
    elapsed, res = await run_with_batch_size(six_tasks, bs)
    ok_count = sum(1 for r in res if r["status"] == "ok")
    print(f"  BATCH_SIZE={bs}: {elapsed:.2f}s total, {ok_count}/{len(six_tasks)} ok")

## Part 13 — Error Injection: Watch Retry + Backoff in Action

To understand retry behavior, we'll simulate failures by wrapping `ainvoke` with a function that intentionally fails for the first N attempts.

This lets us see:
- The attempt counter increment
- Backoff delay between retries
- Final `"ok"` after recovery, or `"error"` if all retries exhausted

In [ ]:
# Instrumented process_task that logs each attempt
async def process_task_verbose(llm: ChatOpenAI, task: str, index: int,
                               force_fail_attempts: int = 0) -> dict:
    """Same as process_task but prints each attempt. force_fail_attempts simulates early failures."""
    for attempt in range(MAX_RETRIES):
        try:
            if attempt < force_fail_attempts:
                raise RuntimeError(f"Simulated failure on attempt {attempt}")
            response = await llm.ainvoke([HumanMessage(content=task)])
            print(f"  [task={index}] attempt={attempt+1} → OK")
            return {
                "index": index, "task": task,
                "answer": response.content.strip(),
                "status": "ok", "attempts": attempt + 1,
            }
        except Exception as exc:
            wait = RETRY_DELAY * (2 ** attempt) if attempt < MAX_RETRIES - 1 else 0
            print(f"  [task={index}] attempt={attempt+1} → FAIL ({exc}) — wait {wait:.1f}s")
            if attempt < MAX_RETRIES - 1:
                await asyncio.sleep(wait)
            else:
                return {
                    "index": index, "task": task,
                    "answer": "", "status": f"error: {exc}", "attempts": attempt + 1,
                }
    return {"index": index, "task": task, "answer": "", "status": "error", "attempts": MAX_RETRIES}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("=== Task that succeeds on first try ===")
r = await process_task_verbose(llm, "What is DNA? One sentence.", 0, force_fail_attempts=0)
print(f"  Result: {r['status']}, attempts={r['attempts']}")

print("\n=== Task that fails once then succeeds ===")
r = await process_task_verbose(llm, "What is DNA? One sentence.", 1, force_fail_attempts=1)
print(f"  Result: {r['status']}, attempts={r['attempts']}")

print("\n=== Task that exhausts all retries ===")
r = await process_task_verbose(llm, "What is DNA? One sentence.", 2, force_fail_attempts=99)
print(f"  Result: {r['status']}, attempts={r['attempts']}")

## Part 14 — Custom Task Lists

The runner is completely general — it accepts any list of prompt strings. You can use it for:
- Bulk classification ("Is this text positive or negative?")
- Data enrichment ("Summarize this article in one sentence: {text}")
- Translation ("Translate to French: {text}")
- Code review ("Review this function for bugs: {code}")

Let's demonstrate with a different task set — batch translation.

In [ ]:
# Custom task list: batch translation
translation_tasks = [
    "Translate to Spanish (one sentence only): The quick brown fox jumps over the lazy dog.",
    "Translate to French (one sentence only): Knowledge is power.",
    "Translate to Japanese (one sentence only): Hello, how are you?",
    "Translate to German (one sentence only): The sun rises in the east.",
    "Translate to Portuguese (one sentence only): Music is the language of the soul.",
]

print(f"Running {len(translation_tasks)} translation tasks in batches of {BATCH_SIZE}\n")
translation_results = await run_batch_agent(translation_tasks)

print("\nTranslations:")
for r in translation_results:
    lang = r['task'].split('to ')[1].split(' ')[0]
    print(f"  {lang:12s}: {r['answer'][:80]}")

## Part 15 — Persisting Results

In production, you'll want to save results to a file or database rather than just printing them. Here's a simple pattern for writing to JSON and CSV.

In [ ]:
import json
import csv
import io

# --- Save to JSON ---
results_json = json.dumps(results, indent=2)
print("JSON output (first result):")
first = json.loads(results_json)[0]
first["answer"] = first["answer"][:60] + "..."
print(json.dumps(first, indent=2))

# --- Save to CSV (in-memory demo) ---
buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=["index", "status", "attempts", "task", "answer"])
writer.writeheader()
for r in results:
    writer.writerow({
        "index": r["index"],
        "status": r["status"],
        "attempts": r["attempts"],
        "task": r["task"][:40],
        "answer": r["answer"][:60],
    })

print("\nCSV output (header + first 3 rows):")
lines = buf.getvalue().strip().split("\n")
for line in lines[:4]:
    print(" ", line)

# To save to disk:
# with open("results.json", "w") as f:
#     json.dump(results, f, indent=2)
# with open("results.csv", "w", newline="") as f:
#     writer = csv.DictWriter(f, fieldnames=["index", "status", "attempts", "task", "answer"])
#     writer.writeheader(); writer.writerows(results)
print("\n(To save to disk, uncomment the file-writing blocks above.)")

## Part 16 — When NOT to Use Batch Processing

Batch processing is the right tool when tasks are **independent**. It breaks down when:

| Situation | Better Pattern |
|-----------|---------------|
| Task B needs Task A's output | Sequential pipeline, LangGraph StateGraph |
| Tasks share mutable state | Sequential or explicit locking |
| One task is much slower than others | Dynamic queue (asyncio.Queue) instead of fixed batches |
| Tasks have hard ordering requirements | Topological sort + sequential execution |
| N is small (1-3 tasks) | `asyncio.gather` without batching — no need for BATCH_SIZE |

**The pattern in this example assumes complete independence between tasks.** Each prompt is self-contained and the LLM's response to one does not affect any other.

In [ ]:
# Contrast: a pipeline where task B depends on task A's output
# This CANNOT be batched — must run sequentially

async def dependent_pipeline(llm: ChatOpenAI, topic: str) -> dict:
    """Two-step pipeline: first get a fact, then get a follow-up question about it."""
    # Step 1: Get a fact
    r1 = await llm.ainvoke([HumanMessage(content=f"Give me one interesting fact about {topic}.")])
    fact = r1.content.strip()

    # Step 2: Ask a follow-up — DEPENDS on step 1's output
    r2 = await llm.ainvoke([HumanMessage(
        content=f"Given this fact: '{fact}' — what is one follow-up question a curious person might ask?"
    )])
    question = r2.content.strip()

    return {"topic": topic, "fact": fact, "follow_up": question}

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
pipeline_result = await dependent_pipeline(llm, "black holes")
print(f"Topic:     {pipeline_result['topic']}")
print(f"Fact:      {pipeline_result['fact'][:100]}")
print(f"Follow-up: {pipeline_result['follow_up'][:100]}")

## Exercises

### Exercise 1 — Increase BATCH_SIZE and measure the effect
Change `BATCH_SIZE` to 9 (all tasks in one batch) and re-run `run_batch_agent`. Compare the wall-clock time to the original `BATCH_SIZE=3` run. Does it get significantly faster? Why or why not?

```python
# Your code here
BATCH_SIZE = 9
# re-run run_batch_agent and time it
```

---

### Exercise 2 — Add jitter to the backoff
Modify `process_task` to add random jitter to the retry delay. The new delay should be:
```python
delay = RETRY_DELAY * (2 ** attempt) + random.uniform(0, 0.5)
```
Why is jitter useful in production systems?

---

### Exercise 3 — Retry only failed tasks
After running `run_batch_agent`, extract only the failed results and re-run the batch runner on just those tasks. This "retry pass" pattern is useful for handling transient API errors without reprocessing all tasks.

```python
# Your code here
failed_tasks = # extract tasks where status != "ok"
retry_results = # re-run only those
```

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 1: Increase BATCH_SIZE to 9
# ============================================================
import time

async def run_with_bs(tasks, bs):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    results = []
    batches = [tasks[i:i+bs] for i in range(0, len(tasks), bs)]
    t0 = time.time()
    for bi, batch in enumerate(batches):
        batch_results = await asyncio.gather(
            *[process_task(llm, task, i + bi * bs) for i, task in enumerate(batch)]
        )
        results.extend(batch_results)
    return time.time() - t0, results

elapsed_9, _ = await run_with_bs(SAMPLE_TASKS, 9)
elapsed_3, _ = await run_with_bs(SAMPLE_TASKS, 3)

print(f"BATCH_SIZE=3: {elapsed_3:.2f}s")
print(f"BATCH_SIZE=9: {elapsed_9:.2f}s")
print(f"Speedup: {elapsed_3/elapsed_9:.2f}x")
print("\nNote: BATCH_SIZE=9 fires all requests simultaneously. Speedup is real but")
print("diminishing — network latency dominates once you're past ~5 concurrent calls.")

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 2: Add jitter to backoff
# ============================================================
import random

async def process_task_with_jitter(llm: ChatOpenAI, task: str, index: int) -> dict:
    for attempt in range(MAX_RETRIES):
        try:
            response = await llm.ainvoke([HumanMessage(content=task)])
            return {
                "index": index, "task": task,
                "answer": response.content.strip(),
                "status": "ok", "attempts": attempt + 1,
            }
        except Exception as exc:
            if attempt < MAX_RETRIES - 1:
                # Jitter: base exponential delay + random offset
                delay = RETRY_DELAY * (2 ** attempt) + random.uniform(0, 0.5)
                print(f"  [task={index}] attempt {attempt+1} failed, retrying in {delay:.2f}s")
                await asyncio.sleep(delay)
            else:
                return {
                    "index": index, "task": task,
                    "answer": "", "status": f"error: {exc}", "attempts": attempt + 1,
                }
    return {"index": index, "task": task, "answer": "", "status": "error", "attempts": MAX_RETRIES}

# Verify jitter schedule differs across calls
print("Jitter effect — same config, different delays:")
for trial in range(3):
    delays = [RETRY_DELAY * (2**a) + random.uniform(0, 0.5) for a in range(2)]
    print(f"  Trial {trial+1}: {[f'{d:.3f}s' for d in delays]}")

print("\nWhy jitter? Without it, all clients failing at the same moment retry at")
print("exactly 1s, 2s, 4s — creating synchronized bursts (thundering herd).")
print("Jitter spreads retries across a time window, smoothing server load.")

In [ ]:
# ============================================================
# ANSWER KEY — Exercise 3: Retry only failed tasks
# ============================================================

# Simulate a first run with some injected failures
async def run_with_failures(tasks: list[str], fail_indices: set) -> list[dict]:
    """run_batch_agent variant that injects failures at specific indices."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    results = []
    batches = [tasks[i:i + BATCH_SIZE] for i in range(0, len(tasks), BATCH_SIZE)]
    for bi, batch in enumerate(batches):
        coros = []
        for i, task in enumerate(batch):
            global_idx = i + bi * BATCH_SIZE
            if global_idx in fail_indices:
                # Replace task with a prompt that will time out / error in test
                coros.append(process_task_verbose(llm, task, global_idx, force_fail_attempts=99))
            else:
                coros.append(process_task(llm, task, global_idx))
        batch_results = await asyncio.gather(*coros)
        results.extend(batch_results)
    return results

print("First run (tasks 2 and 5 forced to fail):")
first_run = await run_with_failures(SAMPLE_TASKS, fail_indices={2, 5})

ok_first = [r for r in first_run if r["status"] == "ok"]
failed_first = [r for r in first_run if r["status"] != "ok"]
print(f"  {len(ok_first)} succeeded, {len(failed_first)} failed")

# Extract just the failed tasks for retry
failed_tasks_for_retry = [r["task"] for r in failed_first]

print(f"\nRetry pass ({len(failed_tasks_for_retry)} failed tasks):")
retry_results = await run_batch_agent(failed_tasks_for_retry)

ok_retry = [r for r in retry_results if r["status"] == "ok"]
print(f"  Retry recovered: {len(ok_retry)}/{len(failed_tasks_for_retry)}")

# Merge results: replace failed entries with retry results
# (In production, match by task text or a unique task ID)
retry_map = {r["task"]: r for r in retry_results}
final_results = [
    retry_map.get(r["task"], r) if r["status"] != "ok" else r
    for r in first_run
]
final_ok = sum(1 for r in final_results if r["status"] == "ok")
print(f"\nFinal merged results: {final_ok}/{len(final_results)} ok")

---

## Workshop Complete

You have built and explored a production-ready async batch agent runner from the ground up. Key takeaways:

- **`asyncio.gather`** fires N coroutines concurrently and returns results in order — the core of the speedup
- **`ainvoke`** is the async counterpart to `invoke` — use it inside `async def` functions to enable concurrency
- **Batching** (rather than all-at-once) caps concurrency to stay within API rate limits
- **Exponential backoff with jitter** handles transient errors without thundering-herd retry storms
- **Structured result dicts** (`status`, `attempts`, `answer`) make post-processing and retry passes straightforward
- **Independence assumption**: batch processing only works when tasks don't depend on each other's output

---

Next: **example 73** — explore more advanced agent patterns in the series.